# Wood NIR CNN Embedding Experiment

Colab Web???notebook??????GitHub clone/pull?data?????????????????????

Direct Colab URL:
https://colab.research.google.com/github/2Kentaro1/wood-moisture-model-cnn/blob/main/notebooks/nir_cnn_embedding_experiment.ipynb


## 1. Setup

Public GitHub repo?clone/pull??repo root???????


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/2Kentaro1/wood-moisture-model-cnn.git"
REPO_NAME = Path(REPO_URL).stem

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'requirements.txt').exists() and (path / 'src').exists():
            return path
    colab_repo = Path('/content') / REPO_NAME
    if colab_repo.exists() and (colab_repo / 'requirements.txt').exists():
        return colab_repo
    return None

repo_root = find_repo_root()
if repo_root is None:
    subprocess.run(['git', 'clone', REPO_URL, f'/content/{REPO_NAME}'], check=True)
    repo_root = Path('/content') / REPO_NAME
else:
    try:
        subprocess.run(['git', '-C', str(repo_root), 'pull'], check=True)
    except Exception as e:
        print('git pull skipped:', e)

os.chdir(repo_root)
Path('data').mkdir(exist_ok=True)
print('repo root:', Path.cwd())
print('data dir:', Path.cwd() / 'data')


## 2. Install Requirements

fresh runtime?? `INSTALL_REQUIREMENTS=True` ????????????


In [ ]:
INSTALL_REQUIREMENTS = True
if INSTALL_REQUIREMENTS:
    assert Path('requirements.txt').exists(), f"requirements.txt not found in {Path.cwd()}"
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


## 3. Data Check

`train.csv` ? `test.csv` ?clone???repo??? `data/` ??????????Colab???????????????OK???


In [ ]:
DATA_DIR = Path('data')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
print('train.csv:', TRAIN_PATH.exists(), TRAIN_PATH)
print('test.csv:', TEST_PATH.exists(), TEST_PATH)
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    print('Upload train.csv and test.csv to:', DATA_DIR.resolve())


## 4. Experiment Config

??????epochs?device???????


In [ ]:
TRAIN_PATH = 'data/train.csv'
TEST_PATH = 'data/test.csv'
OUTPUT_DIR = 'outputs/nir_cnn_embedding'
BAND = 'full'
EMBEDDING_DIM = 16
TARGET_TRANSFORM = 'none'
EPOCHS = 100
BATCH_SIZE = 64
DEVICE = 'cuda'  # or cpu


## 5. Run Experiment


In [ ]:
assert Path(TRAIN_PATH).exists(), f'Missing train file: {TRAIN_PATH}'
assert Path(TEST_PATH).exists(), f'Missing test file: {TEST_PATH}'

cmd = [
    sys.executable,
    'scripts/run_cnn_embedding_experiment.py',
    '--train-path', TRAIN_PATH,
    '--test-path', TEST_PATH,
    '--output-dir', OUTPUT_DIR,
    '--band', BAND,
    '--embedding-dim', str(EMBEDDING_DIM),
    '--target-transform', TARGET_TRANSFORM,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 6. Compare Results and Submission Files


In [ ]:
import pandas as pd
from IPython.display import display

output_dir = Path(OUTPUT_DIR)
compare_path = output_dir / 'results' / 'final_compare_df.csv'
if not compare_path.exists():
    print(f'Not found: {compare_path}')
    print('Run the experiment cell first. If it failed, inspect that cell traceback/log.')
    print('Existing output files:')
    for path in sorted(output_dir.glob('**/*')):
        if path.is_file():
            print(path)
else:
    display(pd.read_csv(compare_path))
    print('Submission files:')
    for path in sorted((output_dir / 'submissions').glob('*.csv')):
        print(path)


## 7. Analyze Figures and Error Tables


In [ ]:
from IPython.display import Image

results_dir = Path(OUTPUT_DIR) / 'results'
figures_dir = Path(OUTPUT_DIR) / 'figures'

for path in sorted(results_dir.glob('cnn_species_rmse_*.csv')):
    print(path.name)
    display(pd.read_csv(path))

for path in sorted(results_dir.glob('cnn_mc_band_rmse_*.csv')):
    print(path.name)
    display(pd.read_csv(path))

for pattern in [
    'actual_vs_pred_by_species_*.png',
    'embedding_pca_*.png',
    'test_prediction_distribution_*.png',
    'embedding_feature_correlation_*.png',
]:
    matches = sorted(figures_dir.glob(pattern))
    if matches:
        print(matches[-1])
        display(Image(filename=str(matches[-1])))


## SIGNATE submission??

`final_compare_df.csv` ? `overall_rmse` ? `species_mean_rmse` ???????? `submission_phase7_cnn_embedding_extra.csv`?`submission_phase7_stable4_cnn_embedding_extra.csv`?`submission_ensemble_cnn_phase7.csv` ?????????????????
